# Module 6: Autoregressive Decoding & Sampling Strategies

## Learning Objectives
- Implement greedy, top-k, top-p, temperature sampling
- Understand streaming output via SSE
- Compare sampling strategies and their effects
- Connect to Kyro's `Worker::sample()`

## Concept: Autoregressive Generation

LLMs generate tokens one at a time, conditioned on all previous tokens:
$$P(w_t | w_{<t}) = \text{softmax}(\text{lm_head}(h_t))$$

Sampling strategies control the trade-off between determinism and creativity.

**Kyro mapping**: `src/worker.rs::sample()` handles temperature, top-p, and greedy sampling.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def greedy_sample(logits):
    return torch.argmax(logits, dim=-1)

def temperature_sample(logits, temperature=1.0):
    scaled = logits / max(temperature, 1e-10)
    probs = F.softmax(scaled, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze()

def top_k_sample(logits, k=10, temperature=1.0):
    scaled = logits / max(temperature, 1e-10)
    probs = F.softmax(scaled, dim=-1)
    top_k_vals, top_k_idx = torch.topk(probs, k)
    filtered = torch.zeros_like(probs)
    filtered.scatter_(1, top_k_idx, top_k_vals)
    filtered_probs = filtered / filtered.sum(dim=-1, keepdim=True)
    return torch.multinomial(filtered_probs, num_samples=1).squeeze()

def nucleus_sample(logits, p=0.9, temperature=1.0):
    scaled = logits / max(temperature, 1e-10)
    probs = F.softmax(scaled, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    mask = cumulative > p
    sorted_probs[mask] = 0
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
    filtered = torch.zeros_like(probs)
    filtered.scatter_(1, sorted_idx, sorted_probs)
    return torch.multinomial(filtered, num_samples=1).squeeze()

# Test with random logits
torch.manual_seed(42)
logits = torch.randn(10)  # vocab_size=10
print('Logits:', logits.tolist())
print(f'Greedy: {greedy_sample(logits).item()}')
print(f'Temp=0.5: {temperature_sample(logits, 0.5).item()}')
print(f'Top-k=3: {top_k_sample(logits, 3).item()}')
print(f'Nucleus p=0.9: {nucleus_sample(logits).item()}')

## Step 1: Visualize Sampling Distributions

In [ ]:
# Generate logits and visualize sampling distributions
logits = torch.randn(20) * 2
temperatures = [0.3, 0.7, 1.0, 1.5]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for idx, temp in enumerate(temperatures):
    ax = axes[idx // 2][idx % 2]
    scaled = logits / temp
    probs = F.softmax(scaled, dim=-1)
    ax.bar(range(len(probs)), probs.detach().numpy())
    ax.set_title(f'Temperature={temp}')
    ax.set_xlabel('Token ID')
    ax.set_ylabel('Probability')
plt.tight_layout()
plt.savefig('sampling_distributions.png', dpi=100)
plt.show()
print('Saved sampling_distributions.png')

## Step 2: Streaming Output (SSE)

In [ ]:
import time

def generate_stream(logits_fn, max_tokens=10, temperature=1.0):
    """Simulate streaming generation with SSE-like behavior."""
    generated = []
    for step in range(max_tokens):
        logits = logits_fn(step)
        token = temperature_sample(logits, temperature)
        generated.append(token.item())
        yield token.item()

# Simulate a simple logits function
def simple_logits(step):
    torch.manual_seed(42 + step)
    return torch.randn(20)

print('Streaming generation:')
for token in generate_stream(simple_logits, max_tokens=8, temperature=0.7):
    print(f'  Generated token: {token}', flush=True)
    time.sleep(0.1)

print('\nIn Kyro: src/api/openai.rs streams via SSE using axum::response::Sse')
    tokens are sent through token_sender channel in src/worker.rs')

## Exercises
1. Implement beam search.
2. Visualize probability distributions after each sampling step.
3. Explore Kyro's `src/api/openai.rs::chat_completions` streaming and `src/worker.rs::sample()`.